## Evualuacion de los modelos

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import balanced_accuracy_score, classification_report
from utils.model_utils import load_multiple_models
from constants import models_dir, converted_dir, scaled_dir, predicted_dir,separated_dir
import os


Cargamos los datos

In [2]:
X_val = pd.read_csv(f"{separated_dir}/X_val.csv")
X_val_scaled = pd.read_csv(f"{scaled_dir}/X_val.csv")
y_val = pd.read_csv(f"{separated_dir}/y_val.csv")

X_test = pd.read_csv(f"{separated_dir}/X_test.csv")
X_test_scaled = pd.read_csv(f"{scaled_dir}/X_test.csv")

Cargamos los modelos

In [3]:
models_original = load_multiple_models(["rf1", "rf2", "rf3"], models_dir)
models_scaled = load_multiple_models(["rf4", "rf5", "rf6"], models_dir)

Procedemos a evaular los modelos con balanced_accuracy_score

In [4]:
all_results = {
    "Original Models": {},
    "Scaled Models": {}
}


evualamos los modelos originales (datos no normalizados)

In [ ]:
print("\nEvaluating Original Models:")
print("=" * 50)
for name, model in models_original.items():
    print(f"\nEvaluating {name}:")
    print("-" * 50)
    y_pred = model.predict(X_val)
    balanced_acc = balanced_accuracy_score(y_val, y_pred)
    all_results["Original Models"][name] = {
        "balanced_accuracy": balanced_acc,
        "classification_report": classification_report(y_val, y_pred)
    }
    print(f"Balanced Accuracy: {balanced_acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_val, y_pred))

Evualamos los modelos que trabajan con los datos normalizados.

In [ ]:
print("\nEvaluating Scaled Models:")
print("=" * 50)
for name, model in models_scaled.items():
    print(f"\nEvaluating {name}:")
    print("-" * 50)
    y_pred = model.predict(X_val_scaled)
    balanced_acc = balanced_accuracy_score(y_val, y_pred)
    all_results["Scaled Models"][name] = {
        "balanced_accuracy": balanced_acc,
        "classification_report": classification_report(y_val, y_pred)
    }
    print(f"Balanced Accuracy: {balanced_acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_val, y_pred))



Calculamos el mejor modelo

In [ ]:
best_acc = 0
best_model_name = None
best_model_type = None

for model_type in all_results:
    for model_name in all_results[model_type]:
        curr_acc = all_results[model_type][model_name]["balanced_accuracy"]
        if curr_acc > best_acc:
            best_acc = curr_acc
            best_model_name = model_name
            best_model_type = model_type

print(f"\nBest Model Overall: {best_model_name} from {best_model_type}")
print(f"Best Balanced Accuracy: {best_acc:.4f}")

Generamos las predicciones con el mejor modelo

In [ ]:
if best_model_type == "Original Models":
    best_model = models_original[best_model_name]
    X_test_final = X_test
else:
    best_model = models_scaled[best_model_name]
    X_test_final = X_test_scaled

# Make predictions
y_test_pred = best_model.predict(X_test_final)

# Save predictions
os.makedirs(predicted_dir, exist_ok=True)
pd.DataFrame(y_test_pred, columns=['predicted']).to_csv(
    f"{predicted_dir}/vehiculos_test_preds.csv", 
    index=False
)

print(f"\nPredictions saved to: {predicted_dir}/vehiculos_test_preds.csv")